# Eligibility Criteria Data Prep

Fetches raw eligibility text from ClinicalTrials.gov API v2 for every trial in the judged pool,
parses inclusion and exclusion criterion sentences with ctproc's `process_eligibility_naive`,
and saves `criteria_data.jsonl` to Drive.

**Output format** (one record per trial):
```json
{"nct_id": "NCT001234", "include_criteria": ["crit A", "crit B"], "exclude_criteria": ["crit X"]}
```

**Used by:** `rerank_criteria.ipynb` (Tier 3a criterion-level reranker).

**Estimated runtime:** ~10–20 min depending on number of judged trials (~2k) and API latency.
Run once; subsequent Tier 3a runs load from Drive.

In [ ]:
!pip install -q git+https://github.com/semajyllek/ctproc.git
!pip install -q requests tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
DATA_ROOT    = '/content/drive/MyDrive/ct_data23'
QRELS_PATH   = f'{DATA_ROOT}/unified_qrels.jsonl'
OUTPUT_PATH  = f'{DATA_ROOT}/criteria_data.jsonl'

CT_API_BASE  = 'https://clinicaltrials.gov/api/v2/studies'
MAX_RETRIES  = 3
REQ_INTERVAL = 0.15   # seconds between requests (~7 req/s, well under the 10/s limit)

In [ ]:
import json

nct_ids = set()
with open(QRELS_PATH) as f:
    for line in f:
        rec = json.loads(line)
        nct_ids.add(rec['doc_id'])

nct_ids = sorted(nct_ids)
print(f'Unique judged NCT IDs: {len(nct_ids)}')
print(f'Sample: {nct_ids[:5]}')

In [ ]:
# Sanity-check the API with one trial before the full fetch
import requests

def _get_elig_text(data: dict) -> str:
    return (
        data
        .get('protocolSection', {})
        .get('eligibilityModule', {})
        .get('eligibilityCriteria', '')
    )

test_id = nct_ids[0]
resp = requests.get(f'{CT_API_BASE}/{test_id}', timeout=20)
resp.raise_for_status()
sample_text = _get_elig_text(resp.json())

print(f'Test NCT ID: {test_id}')
print(f'Field present: {bool(sample_text)}')
print(f'First 500 chars:\n{sample_text[:500]}')

In [ ]:
import time
from tqdm.auto import tqdm

def fetch_elig_text(nct_id: str) -> str | None:
    """Fetch raw eligibility text for a single trial. Returns None on persistent failure."""
    for attempt in range(MAX_RETRIES):
        try:
            resp = requests.get(f'{CT_API_BASE}/{nct_id}', timeout=20)
            resp.raise_for_status()
            return _get_elig_text(resp.json())
        except Exception as e:
            if attempt == MAX_RETRIES - 1:
                print(f'  FAILED {nct_id}: {e}')
                return None
            time.sleep(2 ** attempt)
    return None

elig_texts: dict[str, str] = {}
for nct_id in tqdm(nct_ids, desc='Fetching'):
    result = fetch_elig_text(nct_id)
    elig_texts[nct_id] = result if result is not None else ''
    time.sleep(REQ_INTERVAL)

n_failed  = sum(1 for v in elig_texts.values() if v is None)
n_empty   = sum(1 for v in elig_texts.values() if not v)
print(f'\nFetched {len(elig_texts)} trials — empty: {n_empty}, failed: {n_failed}')

In [ ]:
import re
from ctproc.eligibility import process_eligibility_naive

def normalize_elig_bullets(text: str) -> str:
    """API v2 uses '* ' bullets; process_eligibility_naive expects '- '. Convert."""
    return re.sub(r'^\* ', '- ', text, flags=re.MULTILINE)

records = []
n_no_text = 0
for nct_id in nct_ids:
    elig_text = elig_texts.get(nct_id, '')
    if not elig_text:
        n_no_text += 1
        records.append({'nct_id': nct_id, 'include_criteria': [], 'exclude_criteria': []})
        continue
    inc, exc = process_eligibility_naive(normalize_elig_bullets(elig_text))
    records.append({'nct_id': nct_id, 'include_criteria': inc, 'exclude_criteria': exc})

inc_lens = [len(r['include_criteria']) for r in records]
exc_lens = [len(r['exclude_criteria']) for r in records]
print(f'Parsed {len(records)} trials ({n_no_text} with no eligibility text)')
print(f'Inclusion criteria/trial  — min: {min(inc_lens)}  mean: {sum(inc_lens)/len(inc_lens):.1f}  max: {max(inc_lens)}')
print(f'Exclusion criteria/trial  — min: {min(exc_lens)}  mean: {sum(exc_lens)/len(exc_lens):.1f}  max: {max(exc_lens)}')

In [ ]:
# Spot-check a few parsed records
for r in records[:3]:
    print(f"\n=== {r['nct_id']} ===")
    print(f"  Inc ({len(r['include_criteria'])}): {r['include_criteria'][:3]}")
    print(f"  Exc ({len(r['exclude_criteria'])}): {r['exclude_criteria'][:3]}")

In [ ]:
with open(OUTPUT_PATH, 'w') as f:
    for r in records:
        json.dump(r, f)
        f.write('\n')

print(f'Saved {len(records)} records → {OUTPUT_PATH}')